# Credit Card Fraud Detection: Exploratory Data Analysis

This notebook explores the Kaggle credit-card fraud dataset. Run it with the notebook working directory set to `ml-service/notebooks` so `../data/creditcard.csv` points to the raw CSV. The analysis reads the CSV only and does not modify it.

## Load the data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("../data/creditcard.csv")
df = pd.read_csv(DATA_PATH)

print(f"Loaded {DATA_PATH}")
df.head()

## Dataset shape, data types, missing values, and duplicates

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())

print("\nMissing values by column:")
display(df.isna().sum().rename("missing values").to_frame())

print(f"\nDuplicate rows: {df.duplicated().sum():,}")

## Class balance

In [ ]:
class_counts = df["Class"].value_counts().reindex([0, 1], fill_value=0)
class_summary = pd.DataFrame(
    {
        "count": class_counts,
        "percentage": class_counts / len(df) * 100,
    }
).rename(index={0: "Genuine (0)", 1: "Fraud (1)"})
display(class_summary)

ax = class_counts.rename(index={0: "Genuine (0)", 1: "Fraud (1)"}).plot(
    kind="bar", color=["#4C78A8", "#E45756"], figsize=(7, 4)
)
ax.set_yscale("log")
ax.set_title("Transaction counts by class (log scale)")
ax.set_xlabel("Transaction class")
ax.set_ylabel("Number of transactions (log scale)")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Takeaway:** Fraud is a tiny minority class, and the logarithmic axis makes its count visible beside the much larger genuine class.

## Transaction amount by class

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label, name, color in [(0, "Genuine", "#4C78A8"), (1, "Fraud", "#E45756")]:
    amounts = df.loc[df["Class"] == label, "Amount"]
    ax.hist(amounts, bins=60, density=True, alpha=0.55, label=name, color=color)
ax.set_title("Transaction amount distribution by class")
ax.set_xlabel("Amount")
ax.set_ylabel("Density")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Takeaway:** Compare the amount distributions to see whether transaction size alone separates fraud from genuine payments; overlap means amount cannot identify every fraud.

## Transaction time by class

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label, name, color in [(0, "Genuine", "#4C78A8"), (1, "Fraud", "#E45756")]:
    times = df.loc[df["Class"] == label, "Time"]
    ax.hist(times, bins=60, density=True, alpha=0.55, label=name, color=color)
ax.set_title("Transaction time distribution by class")
ax.set_xlabel("Seconds from the first recorded transaction")
ax.set_ylabel("Density")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Takeaway:** The time distributions show when each class appears in this two-day sample, but time patterns should be checked for generalization beyond this period.

## Ten features most correlated with class

In [ ]:
class_correlations = (
    df.corr(numeric_only=True)["Class"]
    .drop("Class")
    .dropna()
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)

ax = class_correlations.plot(kind="barh", color="#72B7B2", figsize=(8, 5))
ax.set_title("Ten features with strongest absolute correlation to Class")
ax.set_xlabel("Absolute Pearson correlation")
ax.set_ylabel("Feature")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Takeaway:** These features have the strongest one-at-a-time linear associations with fraud, but correlation alone does not establish predictive value or causation.

## What this means for modelling

Fraud is rare, so a model that predicts every transaction as genuine can still be correct on almost all rows. That would give high accuracy while missing nearly every fraudulent payment, which is why accuracy is misleading here. We should instead examine precision, recall, F1, ROC-AUC, PR-AUC, and the confusion matrix. Resampling such as SMOTE or class weights can help a model pay more attention to fraud, but apply resampling only to training data (for example, inside a training pipeline) so validation and test results remain a fair measure of performance.